# 04 · Working software can still give biased causal answers

The clean simulator deliberately makes identification easy: measured confounders are available, each angle has residual variation, and the response matches the model family. This notebook adds an unobserved variable that affects both treatment and outcome. We do not supply it to EconML.

Intervals may look narrow even when identification fails. The simulator lets us compare effects with known truth; real data offers no such oracle. No simulation here supplies clinical evidence.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ARTIFACTS = ROOT / "artifacts" / "notebooks"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 42
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False})
print("SYNTHETIC ONLY — no clinical conclusions.")

SYNTHETIC ONLY — no clinical conclusions.


In [2]:
from plan_ml_poc.diagnostics import oracle_diagnostics
from plan_ml_poc.synthetic import make_cases
from plan_ml_poc.training import fit_models, predictive_metrics, split_cases

rows = []
for hidden in [0.0, 1.3]:
    train, test = split_cases(make_cases(2500, SEED, hidden_confounding=hidden), SEED)
    fitted = fit_models(train, SEED)
    rows.append({"hidden_confounding_strength": hidden,
                 **predictive_metrics(fitted, test), **oracle_diagnostics(fitted, test)})
comparison = pd.DataFrame(rows)
display(comparison)

/Users/kevin/kinomatic/plan-ml-poc/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,hidden_confounding_strength,factual_mae,factual_rmse,oracle_effect_mae,fixed_contrast_pointwise_coverage
0,0.0,1.754007,2.137564,0.075724,1.0
1,1.3,2.711720,3.376950,0.922741,0.0


The change in bias or coverage is one simulation realization, not a monotonicity guarantee. Increase sample sizes or repeat seeds as an exercise. Good factual outcome prediction does not establish that changing an angle causes improvement.

## Assumptions to carry forward

- **Consistency:** treatment must describe a well-defined executed intervention. A strategy label does not fully specify component positions or co-interventions.
- **Measured confounding:** include pretreatment causes of both treatment and outcome. More rows do not repair omitted causes.
- **Overlap:** sufficient joint treatment variation is needed for comparable patients. Nearest-neighbor distances are a limited screening proxy.
- **Effect-model specification:** this quadratic-in-treatment, linear-in-X model is a teaching assumption. It does not discover arbitrary 3D biomechanics.
- **No interference / independent units:** this fit assumes independent patient groups and does not provide hospital-cluster-robust inference.
- **Follow-up:** missing or delayed outcomes require a defined analysis; do not replace missing FJS with zero or fabricate labels from plans.
- **Selection:** search-selected pointwise intervals need independent evaluation; the optimizer can amplify model error.

## When actual data arrives

1. Define one procedure, eligible population, decision time, executed treatment, reference and follow-up window.
2. Link pseudonymous patient/joint IDs across imaging, proposed/approved plans, achieved positions and outcomes. Keep these plan stages separate.
3. Audit units, timing, measurement error, absent outcomes, treatment variation and confounders. Fit preprocessing within training folds.
4. Preserve patient groups, reserve chronological and external-site cohorts, and plan sensitivity analyses.
5. Validate physical feasibility with the actual geometry/biomechanics engine and expert-defined constraints.
6. Study supported prespecified contrasts before optimizing. Use shadow-mode/prospective evaluation before patient-facing use.

THA needs its own features, cup/stem actions, constraints and outcome specification. Do not relabel this TKA simulator to claim THA coverage. Revision and complication targets are future separate models, including censoring where relevant.